# SR-IMSRG(3) Flow Equation Derivation

This notebook uses `qcombo.easyCombo` with `wick_mode='SR'` to derive the **SR-IMSRG(3) flow equations**.

## Theoretical Background

In the IMSRG (In-Medium Similarity Renormalization Group) framework, the Hamiltonian $H(s)$ evolves with the flow parameter $s$ according to:

$$\frac{dH(s)}{ds} = [\eta(s), H(s)]$$

where $\eta(s)$ is the **generator** (here we use the Brillouin generator $\eta = [H, A]$).

### SR vs MR

**SR (Single-Reference)** mode differs from MR (Multi-Reference) in two key ways:
- **No multi-body $\lambda$**: Only 1-body density matrices ($\lambda^a_b$, $\xi^a_b$) are allowed. There are no $\lambda^{ab}_{cd}$ or higher-body contractions.
- **Contraction rules**: $[m,n] \to k$ only if $|m-n| \leq k < m+n$. For example, $[1,2]$ can only produce 1-body and 2-body terms, not 0-body.

### IMSRG(3) Truncation

At the **IMSRG(3)** truncation level, we keep operators up to 3-body:

$$H(s) = E(s) + \sum_{ij} f^i_j(s) \{a^\dagger_i a_j\} + \frac{1}{4}\sum_{ijkl} \Gamma^{ij}_{kl}(s) \{a^\dagger_i a^\dagger_j a_l a_k\} + \frac{1}{36}\sum_{ijklmn} W^{ijk}_{lmn}(s) \{a^\dagger_i a^\dagger_j a^\dagger_k a_n a_m a_l\}$$

The flow equations are:
- **0-body flow** ($dE/ds$): $[\eta, H]_{0B}$
- **1-body flow** ($df/ds$): $[\eta, H]_{1B}$
- **2-body flow** ($d\Gamma/ds$): $[\eta, H]_{2B}$
- **3-body flow** ($dW/ds$): $[\eta, H]_{3B}$


**Reference**: H. Heinz, et.al **In-medium similarity renormalization group with three-body operators**


In [1]:
# Import qcombo and necessary utility functions
import qcombo
from IPython.display import display, Latex
from sympy import IndexedBase, symbols
from sympy import preorder_traversal
from sympy.tensor.indexed import Indexed

import time
from qcombo.simplify import filterLambdaBody

# Define tensor symbols
A = IndexedBase('A')       # generator
G = IndexedBase('G')       # left operator (in easyCombo)
H = IndexedBase('H')       # right operator (in easyCombo)
f = IndexedBase('f')       # one-body matrix element
Gamma = IndexedBase(915)   # Γ two-body matrix element
W = IndexedBase('W')       # W three-body matrix element
lamda = IndexedBase(955)   # λ (lambda) density matrix
n = IndexedBase('n')       # occupation number
eta = IndexedBase(951)     # η generator matrix element (1B)
eta2 = IndexedBase(952)    # η two-body generator matrix element
eta3 = IndexedBase(953)    # η three-body generator matrix element

print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


In [2]:
# Helper function: display SymPy expressions as LaTeX in Jupyter
def jupyterDisplay(expr, title=None):
    """
    Display SymPy expression in LaTeX format in Jupyter Notebook
    """
    if expr == 0 or expr is None:
        display(Latex(f"$$0$$"))
        return
    latex_expr = qcombo.texExp(expr)
    if title:
        print(title)
    display(Latex(f"$${latex_expr}$$"))

# Utility: find tensor in expression
def find_H_tensor(expr):
    """Find the H tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return term
    return None

def find_G_tensor(expr):
    """Find the G tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            return term
    return None

def find_A_tensor(expr):
    """Find the A tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('A'):
            return term
    return None

def replace_G_Base(expr, newBase):
    """Replace the base of all G tensors with a new base"""
    G_tensor = find_G_tensor(expr)
    if G_tensor is None:
        return expr
    else:
        return expr.xreplace({G_tensor.base: IndexedBase(newBase)})

def replace_H_Base(expr, newBase):
    """Replace the base of all H tensors with a new base"""
    G_tensor = find_H_tensor(expr)
    if G_tensor is None:
        return expr
    else:
        return expr.xreplace({G_tensor.base: IndexedBase(newBase)})

def re_antisymmetry(expr):
    """
    Re-index the expression and restore index antisymmetry.
    e.g. A[i,j]*B[k,l] -> A[a,b]*B[c,d]
    """
    canon_expr = qcombo.canonical.canonicalize(expr.expand(),parallel=False,show_process=False)
    reIndices_expr,indices_set = qcombo.tools.indicesMultToSimp(canon_expr,parallel=False,show_process=False)
    re_AntiSymetry_expr = qcombo.tools.antisymmetrize_expr(reIndices_expr)
    simplified_expr = qcombo.simplifyUseBoth(re_AntiSymetry_expr.expand(),show_process=False,parallel=False)
    united_expr = qcombo.MergeSameMatrixElement(simplified_expr)
    return united_expr

print("All helper functions defined.")

All helper functions defined.


---
## 1. Zero-body Flow: $dE/ds$

The 0-body flow equation comes from $[\eta, H]_{0B}$.




In [4]:
# dE/ds comes from [1,1]_0, [2,2]_0, [3,3]_0 (SR-allowed)

print("="*60)
print("Computing zero-body flow (SR mode)...")
print("="*60)

t0 = time.time()
comm_110 = qcombo.easyCombo(1, 1, 0, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_220 = qcombo.easyCombo(2, 2, 0, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_330 = qcombo.easyCombo(3, 3, 0, wick_mode='SR', parallel=True, show_process=True, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing zero-body flow (SR mode)...
Parallel computing with 7063 tasks...
parallel wick: [██████████████████████████████████████████████████] [7063/7063]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:16.7s
Parallel computing completed!
Parallel computing with 7063 tasks...
parallel wick: [██████████████████████████████████████████████████] [7063/7063]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:18.7s
Parallel computing completed!
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 0-body terms
Parallel filtering 7140 terms...
parallel filtering: [██████████████████████████████████████████████████] [7140/7140]100.0%  | Remaining: 0.0s
parallel filtering completed! Total Time:8.2s
Parallel canonicalizing 504 terms...
parallel canonicalizing: [██████████████████████████████████████████████████] [504/504]100.0%  | Remaining: 0.0s
parallel canonicalizing completed! Total Time:4.1s
Parallel repetitive index simplifying 504 te

In [ ]:
# Aggregate results from expr_dict and display raw expressions
comm_110_expr = 0
comm_220_expr = 0
comm_330_expr = 0

for key, value in comm_110.expr_dict.items():
    comm_110_expr += value

for key, value in comm_220.expr_dict.items():
    comm_220_expr += value

for key, value in comm_330.expr_dict.items():
    comm_330_expr += value

# print('commutator [1,1]-0 expression:')
# jupyterDisplay(comm_110_expr)
# print('commutator [2,2]-0 expression:')
# jupyterDisplay(comm_220_expr)
# print('commutator [3,3]-0 expression:')
# jupyterDisplay(comm_330_expr)


In [ ]:
# Restore index antisymmetry of the expressions
comm_110_expr = re_antisymmetry(comm_110_expr)
comm_220_expr = re_antisymmetry(comm_220_expr)
comm_330_expr = re_antisymmetry(comm_330_expr)

# print('commutator [1,1]-0 expression (antisymmetrized):')
# jupyterDisplay(comm_110_expr)
# print('commutator [2,2]-0 expression (antisymmetrized):')
# jupyterDisplay(comm_220_expr)
# print('commutator [3,3]-0 expression (antisymmetrized):')
# jupyterDisplay(comm_330_expr)


In [ ]:
# Replace G → η (generator), H → f/Γ/W to obtain the flow equation
# with proper coefficients
dE_ds_110_expr = replace_H_Base(replace_G_Base(comm_110_expr, r'\eta'), "f")
dE_ds_220_expr = replace_H_Base(replace_G_Base(comm_220_expr, r'\eta'), r'\Gamma')/16
dE_ds_330_expr = replace_H_Base(replace_G_Base(comm_330_expr, r'\eta'), "W")/36

# jupyterDisplay(dE_ds_110_expr, "[η_1B, f] → 0B:")
# jupyterDisplay(dE_ds_220_expr, "[η_2B, Γ] → 0B:")
# jupyterDisplay(dE_ds_330_expr, "[η_3B, W] → 0B:")


In [ ]:
print("dE/ds flow equation contains:")
display(Latex(r'$$d E =  $$'))
# [1,1]-0
print('commutator [1,1]-0, lambda_1B:')
jupyterDisplay(dE_ds_110_expr)
# [2,2]-0 — classify by λ body number (SR: only λ_1B)
print('commutator [2,2]-0, lambda_1B:')
jupyterDisplay(dE_ds_220_expr)
# [3,3]-0 — classify by λ body number
print('commutator [3,3]-0, lambda_1B:')
jupyterDisplay(dE_ds_330_expr)


dE/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,1]-0, lambda_1B:


<IPython.core.display.Latex object>

commutator [2,2]-0, lambda_1B:


<IPython.core.display.Latex object>

commutator [3,3]-0, lambda_1B:


<IPython.core.display.Latex object>

# **dE/ds** in Reference

$$\frac{dE}{dS} = \sum_{pq} (n_p \bar{n}_q - \bar{n}_p n_q) \eta_{pq} f_{qp} + \frac{1}{4} \sum_{pqrs} (n_p n_q \bar{n}_r \bar{n}_s - \bar{n}_p \bar{n}_q n_r n_s) \eta_{pqrs} \Gamma_{rspq}$$
$$+ \frac{1}{36} \sum_{pqrstu} (n_p n_q n_r \bar{n}_s \bar{n}_t \bar{n}_u - \bar{n}_p \bar{n}_q \bar{n}_r n_s n_t n_u) \eta_{pqrstu} W_{stupqr},$$

---
## 2. One-body Flow: $df/ds$

The 1-body flow equation comes from $[\eta, H]_{1B}$.



In [ ]:
# df/ds comes from [1,1]_1, [1,2]_1, [2,1]_1, [2,2]_1, [2,3]_1, [3,2]_1, [3,3]_1 (SR-allowed)

print("="*60)
print("Computing one-body flow (SR mode)...")
print("="*60)

t0 = time.time()
comm_111 = qcombo.easyCombo(1, 1, 1, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_121 = qcombo.easyCombo(1, 2, 1, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_211 = qcombo.easyCombo(2, 1, 1, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_221 = qcombo.easyCombo(2, 2, 1, wick_mode='SR', parallel=False, show_process=False, savefile=False)
# use parallel
comm_231 = qcombo.easyCombo(2, 3, 1, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_321 = qcombo.easyCombo(3, 2, 1, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_331 = qcombo.easyCombo(3, 3, 1, wick_mode='SR', parallel=True, show_process=True, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing one-body flow (SR mode)...
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:2.0s
Parallel computing completed!
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:2.6s
Parallel computing completed!
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 1-body terms
Parallel filtering 624 terms...
parallel filtering: [██████████████████████████████████████████████████] [624/624]100.0%  | Remaining: 0.0s
parallel filtering completed! Total Time:2.4s
Parallel canonicalizing 216 terms...
parallel canonicalizing: [██████████████████████████████████████████████████] [216/216]100.0%  | Remaining: 0.0s
parallel canonicalizing completed! Total Time:1.9s
Parallel repetitive index simplifying 216 terms...
paral

In [ ]:
# Aggregate results from expr_dict and display raw expressions
comm_111_expr = 0
comm_121_expr = 0
comm_211_expr = 0
comm_221_expr = 0
comm_231_expr = 0
comm_321_expr = 0
comm_331_expr = 0

for key, value in comm_111.expr_dict.items():
    comm_111_expr += value
for key, value in comm_121.expr_dict.items():
    comm_121_expr += value
for key, value in comm_211.expr_dict.items():
    comm_211_expr += value
for key, value in comm_221.expr_dict.items():
    comm_221_expr += value
for key, value in comm_231.expr_dict.items():
    comm_231_expr += value
for key, value in comm_321.expr_dict.items():
    comm_321_expr += value
for key, value in comm_331.expr_dict.items():
    comm_331_expr += value

# print('commutator [1,1]-1 expression:')
# jupyterDisplay(comm_111_expr)
# print('commutator [1,2]-1 expression:')
# jupyterDisplay(comm_121_expr)
# print('commutator [2,1]-1 expression:')
# jupyterDisplay(comm_211_expr)
# print('commutator [2,2]-1 expression:')
# jupyterDisplay(comm_221_expr)
# print('commutator [2,3]-1 expression:')
# jupyterDisplay(comm_231_expr)
# print('commutator [3,2]-1 expression:')
# jupyterDisplay(comm_321_expr)
# print('commutator [3,3]-1 expression:')
# jupyterDisplay(comm_331_expr)


In [ ]:
# Restore index antisymmetry of the expressions
comm_111_expr = re_antisymmetry(comm_111_expr)
comm_121_expr = re_antisymmetry(comm_121_expr)
comm_211_expr = re_antisymmetry(comm_211_expr)
comm_221_expr = re_antisymmetry(comm_221_expr)
comm_231_expr = re_antisymmetry(comm_231_expr)
comm_321_expr = re_antisymmetry(comm_321_expr)
comm_331_expr = re_antisymmetry(comm_331_expr)

# print('commutator [1,1]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_111_expr)
# print('commutator [1,2]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_121_expr)
# print('commutator [2,1]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_211_expr)
# print('commutator [2,2]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_221_expr)
# print('commutator [2,3]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_231_expr)
# print('commutator [3,2]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_321_expr)
# print('commutator [3,3]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_331_expr)


In [ ]:
# Replace G → η (generator), H → f/Γ/W to obtain the flow equation
# with proper coefficients
A_tensor = find_A_tensor(comm_111_expr)
# jupyterDisplay(A_tensor, "A tensor:")

df_ds_111_expr = replace_H_Base(replace_G_Base(comm_111_expr, r'\eta'), "f")/A_tensor
df_ds_121_expr = replace_H_Base(replace_G_Base(comm_121_expr, r'\eta'), r'\Gamma')/A_tensor/4
df_ds_211_expr = replace_H_Base(replace_G_Base(comm_211_expr, r'\eta'), "f")/A_tensor/4
df_ds_221_expr = replace_H_Base(replace_G_Base(comm_221_expr, r'\eta'), r'\Gamma')/A_tensor/16
df_ds_231_expr = replace_H_Base(replace_G_Base(comm_231_expr, r'\eta'), "W")/A_tensor/36/4
df_ds_321_expr = replace_H_Base(replace_G_Base(comm_321_expr, r'\eta'), r'\Gamma')/A_tensor/36/4
df_ds_331_expr = replace_H_Base(replace_G_Base(comm_331_expr, r'\eta'), "W")/A_tensor/36/36

# jupyterDisplay(df_ds_111_expr, "[η_1B, f] → 1B:")
# jupyterDisplay(df_ds_121_expr, "[η_1B, Γ] → 1B:")
# jupyterDisplay(df_ds_211_expr, "[η_2B, f] → 1B:")
# jupyterDisplay(df_ds_221_expr, "[η_2B, Γ] → 1B:")
# jupyterDisplay(df_ds_231_expr, "[η_2B, W] → 1B:")
# jupyterDisplay(df_ds_321_expr, "[η_3B, Γ] → 1B:")
# jupyterDisplay(df_ds_331_expr, "[η_3B, W] → 1B:")


In [ ]:
print("df/ds flow equation contains:")
display(Latex(r'$$d f^{a}_{b} =  $$'))
# [1,1]-1
print('commutator [1,1]-1, lambda_1B:')
jupyterDisplay(df_ds_111_expr)
# [1,2]-1
print('commutator [1,2]-1, lambda_1B:')
jupyterDisplay(df_ds_121_expr)
# [2,1]-1
print('commutator [2,1]-1, lambda_1B:')
jupyterDisplay(df_ds_211_expr)
# [2,2]-1
print('commutator [2,2]-1, lambda_1B:')
jupyterDisplay(df_ds_221_expr)
# [2,3]-1
print('commutator [2,3]-1, lambda_1B:')
jupyterDisplay(df_ds_231_expr)
# [3,2]-1
print('commutator [3,2]-1, lambda_1B:')
jupyterDisplay(df_ds_321_expr)
# [3,3]-1
print('commutator [3,3]-1, lambda_1B:')
jupyterDisplay(df_ds_331_expr)


df/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,1]-1, lambda_1B:


<IPython.core.display.Latex object>

commutator [1,2]-1, lambda_1B:


...

<IPython.core.display.Latex object>

commutator [3,2]-1, lambda_1B:


<IPython.core.display.Latex object>

commutator [3,3]-1, lambda_1B:


<IPython.core.display.Latex object>

# **df/ds** in Reference

$$\frac{df_{12}}{ds} = \sum_{p} (\eta_{1p} f_{p2} - f_{1p} \eta_{p2}) + \sum_{pq} (n_{p} \bar{n}_{q} - \bar{n}_{p} n_{q}) (\eta_{pq} \Gamma_{1q2p} - f_{pq} \eta_{1q2p}) $$
$$+ \frac{1}{2} \sum_{pqr} (\bar{n}_{p} \bar{n}_{q} n_{r} + n_{p} n_{q} \bar{n}_{r}) (\eta_{1rpq} \Gamma_{pq2r} - \Gamma_{1rpq} \eta_{pq2r}) $$
$$+ \frac{1}{4} \sum_{pqrs} (n_{p} n_{q} \bar{n}_{r} \bar{n}_{s} - \bar{n}_{p} \bar{n}_{q} n_{r} n_{s}) (\eta_{pqrs} W_{rs1pq2} - \Gamma_{pqrs} \eta_{rs1pq2}) $$
$$+ \frac{1}{12} \sum_{pqrst} (n_{p} n_{q} n_{r} \bar{n}_{s} \bar{n}_{t} + \bar{n}_{p} \bar{n}_{q} \bar{n}_{r} n_{s} n_{t}) (\eta_{st1pqr} W_{pqrst2} - W_{st1pqr} \eta_{pqrst2}),$$

---
## 3. Two-body Flow: $d\Gamma/ds$

The 2-body flow equation comes from $[\eta, H]_{2B}$.



In [ ]:
# dΓ/ds comes from [1,2]_2, [1,3]_2, [2,1]_2, [2,2]_2, [2,3]_2, [3,1]_2, [3,2]_2, [3,3]_2 (SR-allowed)

print("="*60)
print("Computing two-body flow (SR mode)...")
print("="*60)

t0 = time.time()
comm_122 = qcombo.easyCombo(1, 2, 2, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_212 = qcombo.easyCombo(2, 1, 2, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_132 = qcombo.easyCombo(1, 3, 2, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_312 = qcombo.easyCombo(3, 1, 2, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_222 = qcombo.easyCombo(2, 2, 2, wick_mode='SR', parallel=False, show_process=False, savefile=False)
# use parallel
comm_232 = qcombo.easyCombo(2, 3, 2, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_322 = qcombo.easyCombo(3, 2, 2, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_332 = qcombo.easyCombo(3, 3, 2, wick_mode='SR', parallel=True, show_process=True, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing two-body flow (SR mode)...
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:1.9s
Parallel computing completed!
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:1.9s
Parallel computing completed!
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 2-body terms
Parallel filtering 624 terms...
parallel filtering: [██████████████████████████████████████████████████] [624/624]100.0%  | Remaining: 0.0s
parallel filtering completed! Total Time:1.7s
Parallel canonicalizing 288 terms...
parallel canonicalizing: [██████████████████████████████████████████████████] [288/288]100.0%  | Remaining: 0.0s
parallel canonicalizing completed! Total Time:1.7s
Parallel repetitive index simplifying 288 terms...
paral

In [ ]:
# Aggregate results from expr_dict and display raw expressions
comm_122_expr = 0
comm_132_expr = 0
comm_212_expr = 0
comm_222_expr = 0
comm_232_expr = 0
comm_312_expr = 0
comm_322_expr = 0
comm_332_expr = 0

for key, value in comm_122.expr_dict.items():
    comm_122_expr += value
for key, value in comm_132.expr_dict.items():
    comm_132_expr += value
for key, value in comm_212.expr_dict.items():
    comm_212_expr += value
for key, value in comm_222.expr_dict.items():
    comm_222_expr += value
for key, value in comm_232.expr_dict.items():
    comm_232_expr += value
for key, value in comm_312.expr_dict.items():
    comm_312_expr += value
for key, value in comm_322.expr_dict.items():
    comm_322_expr += value
for key, value in comm_332.expr_dict.items():
    comm_332_expr += value

# print('commutator [1,2]-2 expression:')
# jupyterDisplay(comm_122_expr)
# print('commutator [1,3]-2 expression:')
# jupyterDisplay(comm_132_expr)
# print('commutator [2,1]-2 expression:')
# jupyterDisplay(comm_212_expr)
# print('commutator [2,2]-2 expression:')
# jupyterDisplay(comm_222_expr)
# print('commutator [2,3]-2 expression:')
# jupyterDisplay(comm_232_expr)
# print('commutator [3,1]-2 expression:')
# jupyterDisplay(comm_312_expr)
# print('commutator [3,2]-2 expression:')
# jupyterDisplay(comm_322_expr)
# print('commutator [3,3]-2 expression:')
# jupyterDisplay(comm_332_expr)


In [ ]:
# Restore index antisymmetry of the expressions
comm_122_expr = re_antisymmetry(comm_122_expr)
comm_132_expr = re_antisymmetry(comm_132_expr)
comm_212_expr = re_antisymmetry(comm_212_expr)
comm_222_expr = re_antisymmetry(comm_222_expr)
comm_232_expr = re_antisymmetry(comm_232_expr)
comm_312_expr = re_antisymmetry(comm_312_expr)
comm_322_expr = re_antisymmetry(comm_322_expr)
comm_332_expr = re_antisymmetry(comm_332_expr)

# print('commutator [1,2]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_122_expr)
# print('commutator [1,3]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_132_expr)
# print('commutator [2,1]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_212_expr)
# print('commutator [2,2]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_222_expr)
# print('commutator [2,3]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_232_expr)
# print('commutator [3,1]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_312_expr)
# print('commutator [3,2]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_322_expr)
# print('commutator [3,3]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_332_expr)


In [ ]:
# Replace G → η (generator), H → f/Γ/W to obtain the flow equation
# with proper coefficients

A_tensor = find_A_tensor(comm_122_expr)
# jupyterDisplay(A_tensor, "A tensor:")

dG_ds_122_expr = replace_H_Base(replace_G_Base(comm_122_expr, r'\eta'), r'\Gamma')/A_tensor
dG_ds_132_expr = replace_H_Base(replace_G_Base(comm_132_expr, r'\eta'), "W")/A_tensor/9
dG_ds_212_expr = replace_H_Base(replace_G_Base(comm_212_expr, r'\eta'), "f")/A_tensor
dG_ds_222_expr = replace_H_Base(replace_G_Base(comm_222_expr, r'\eta'), r'\Gamma')/A_tensor/4
dG_ds_232_expr = replace_H_Base(replace_G_Base(comm_232_expr, r'\eta'), "W")/A_tensor/36
dG_ds_312_expr = replace_H_Base(replace_G_Base(comm_312_expr, r'\eta'), "f")/A_tensor/9
dG_ds_322_expr = replace_H_Base(replace_G_Base(comm_322_expr, r'\eta'), r'\Gamma')/A_tensor/36
dG_ds_332_expr = replace_H_Base(replace_G_Base(comm_332_expr, r'\eta'), "W")/A_tensor/36/9

# jupyterDisplay(dG_ds_122_expr, "[η_1B, Γ] → 2B:")
# jupyterDisplay(dG_ds_132_expr, "[η_1B, W] → 2B:")
# jupyterDisplay(dG_ds_212_expr, "[η_2B, f] → 2B:")
# jupyterDisplay(dG_ds_222_expr, "[η_2B, Γ] → 2B:")
# jupyterDisplay(dG_ds_232_expr, "[η_2B, W] → 2B:")
# jupyterDisplay(dG_ds_312_expr, "[η_3B, f] → 2B:")
# jupyterDisplay(dG_ds_322_expr, "[η_3B, Γ] → 2B:")
# jupyterDisplay(dG_ds_332_expr, "[η_3B, W] → 2B:")


In [ ]:
from sympy import latex


print("dΓ/ds flow equation contains:")
display(Latex(r'$$d \Gamma^{ab}_{cd} =  $$'))
# [1,2]-2
print('commutator [1,2]-2, lambda_1B:')
jupyterDisplay(dG_ds_122_expr)
# [1,3]-2
print('commutator [1,3]-2, lambda_1B:')
jupyterDisplay(dG_ds_132_expr)
# [2,1]-2
print('commutator [2,1]-2, lambda_1B:')
jupyterDisplay(dG_ds_212_expr)
# [2,2]-2
print('commutator [2,2]-2, lambda_1B:')
jupyterDisplay(dG_ds_222_expr)
# [2,3]-2
print('commutator [2,3]-2, lambda_1B:')
jupyterDisplay(dG_ds_232_expr)
# [3,1]-2
print('commutator [3,1]-2, lambda_1B:')
jupyterDisplay(dG_ds_312_expr)
# [3,2]-2
print('commutator [3,2]-2, lambda_1B:')
jupyterDisplay(dG_ds_322_expr)
# [3,3]-2
print('commutator [3,3]-2, lambda_1B:')
jupyterDisplay(dG_ds_332_expr)


dΓ/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,2]-2, lambda_1B:


<IPython.core.display.Latex object>

commutator [1,3]-2, lambda_1B:


...

<IPython.core.display.Latex object>

commutator [3,2]-2, lambda_1B:


<IPython.core.display.Latex object>

commutator [3,3]-2, lambda_1B:


<IPython.core.display.Latex object>

## $d\Gamma /ds$ in Reference

$$\frac{d\Gamma_{1234}}{ds} = (1 - P_{12}) \sum_{p} (\eta_{1p} \Gamma_{p234} - f_{1p} \eta_{p234}) - (1 - P_{34}) \sum_{p} (\eta_{p3} \Gamma_{12p4} - f_{p3} \eta_{12p4}) $$
$$
+ \frac{1}{2} \sum_{pq} (\bar{n}_{p} \bar{n}_{q} - n_{p} n_{q}) (\eta_{12pq} \Gamma_{pq34} - \Gamma_{12pq} \eta_{pq34}) - (1 - P_{12})(1 - P_{34}) \sum_{pq} (n_{p} \bar{n}_{q} - \bar{n}_{p} n_{q}) \eta_{23q} \Gamma_{1qp4} $$
$$+ \sum_{pq} (n_{p} \bar{n}_{q} - \bar{n}_{p} n_{q}) (\eta_{pq} W_{12q34p} - f_{pq} \eta_{12q34p})$$
$$
+ \frac{1}{2} (1 - P_{12}) \sum_{pqr} (\bar{n}_{p} \bar{n}_{q} n_{r} + n_{p} n_{q} \bar{n}_{r}) (\eta_{r1pq} W_{pq234r} - \Gamma_{r1pq} \eta_{pq234r}) $$
$$
- \frac{1}{2} (1 - P_{34}) \sum_{pqr} (\bar{n}_{p} \bar{n}_{q} n_{r} + n_{p} n_{q} \bar{n}_{r}) (\eta_{pqr3} W_{12rpq4} - \Gamma_{pqr3} \eta_{12rpq4}) $$
$$
+ \frac{1}{6} \sum_{pqrs} (\bar{n}_{p} \bar{n}_{q} \bar{n}_{r} n_{s} - n_{p} n_{q} n_{r} \bar{n}_{s}) (\eta_{12spqr} W_{pqr34s} - W_{12spqr} \eta_{pqr34s}) $$
$$
+ \frac{1}{4} (1 - P_{12})(1 - P_{34}) \sum_{pqrs} (n_{p} n_{q} \bar{n}_{r} \bar{n}_{s} - \bar{n}_{p} \bar{n}_{q} \bar{n}_{r} n_{s}) \eta_{pq1rs3} W_{rs2pq4},$$

---
## 4. Three-body Flow: $dW/ds$

The 3-body flow equation comes from $[\eta, H]_{3B}$.



In [ ]:
# dW/ds comes from [1,3]_3, [2,3]_3, [3,1]_3, [3,2]_3, [3,3]_3 (SR-allowed)


print("="*60)
print("Computing three-body flow (SR mode)...")
print("="*60)

t0 = time.time()
comm_133 = qcombo.easyCombo(1, 3, 3, wick_mode='SR', parallel=False, show_process=False, savefile=False)
comm_313 = qcombo.easyCombo(3, 1, 3, wick_mode='SR', parallel=False, show_process=False, savefile=False)
# use parallel
comm_233 = qcombo.easyCombo(2, 3, 3, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_323 = qcombo.easyCombo(3, 2, 3, wick_mode='SR', parallel=True, show_process=True, savefile=False)
comm_333 = qcombo.easyCombo(3, 3, 3, wick_mode='SR', parallel=True, show_process=True, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing three-body flow (SR mode)...
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:2.5s
Parallel computing completed!
Parallel computing with 552 tasks...
parallel wick: [██████████████████████████████████████████████████] [552/552]100.0%  | Remaining: 0.0s
parallel wick completed! Total Time:2.2s
Parallel computing completed!
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 3-body terms
Parallel filtering 624 terms...
parallel filtering: [██████████████████████████████████████████████████] [624/624]100.0%  | Remaining: 0.0s
parallel filtering completed! Total Time:2.2s
Parallel canonicalizing 108 terms...
parallel canonicalizing: [██████████████████████████████████████████████████] [108/108]100.0%  | Remaining: 0.0s
parallel canonicalizing completed! Total Time:1.3s
Parallel repetitive index simplifying 108 terms...
par

In [ ]:
# Aggregate results from expr_dict and display raw expressions
comm_133_expr = 0
comm_233_expr = 0
comm_313_expr = 0
comm_323_expr = 0
comm_333_expr = 0

for key, value in comm_133.expr_dict.items():
    comm_133_expr += value
for key, value in comm_233.expr_dict.items():
    comm_233_expr += value
for key, value in comm_313.expr_dict.items():
    comm_313_expr += value
for key, value in comm_323.expr_dict.items():
    comm_323_expr += value
for key, value in comm_333.expr_dict.items():
    comm_333_expr += value

# print('commutator [1,3]-3 expression:')
# jupyterDisplay(comm_133_expr)
# print('commutator [2,3]-3 expression:')
# jupyterDisplay(comm_233_expr)
# print('commutator [3,1]-3 expression:')
# jupyterDisplay(comm_313_expr)
# print('commutator [3,2]-3 expression:')
# jupyterDisplay(comm_323_expr)
# print('commutator [3,3]-3 expression:')
# jupyterDisplay(comm_333_expr)


In [ ]:
# Restore index antisymmetry of the expressions
comm_133_expr = re_antisymmetry(comm_133_expr)
comm_233_expr = re_antisymmetry(comm_233_expr)
comm_313_expr = re_antisymmetry(comm_313_expr)
comm_323_expr = re_antisymmetry(comm_323_expr)
comm_333_expr = re_antisymmetry(comm_333_expr)

# print('commutator [1,3]-3 expression (antisymmetrized):')
# jupyterDisplay(comm_133_expr)
# print('commutator [2,3]-3 expression (antisymmetrized):')
# jupyterDisplay(comm_233_expr)
# print('commutator [3,1]-3 expression (antisymmetrized):')
# jupyterDisplay(comm_313_expr)
# print('commutator [3,2]-3 expression (antisymmetrized):')
# jupyterDisplay(comm_323_expr)
# print('commutator [3,3]-3 expression (antisymmetrized):')
# jupyterDisplay(comm_333_expr)


In [ ]:
# Replace G → η (generator), H → f/Γ/W to obtain the flow equation
# with proper coefficients
A_tensor = find_A_tensor(comm_133_expr)
# jupyterDisplay(A_tensor, "A tensor:")

dW_ds_133_expr = replace_H_Base(replace_G_Base(comm_133_expr, r'\eta'), "W")/A_tensor
dW_ds_233_expr = replace_H_Base(replace_G_Base(comm_233_expr, r'\eta'), "W")/A_tensor/4
dW_ds_313_expr = replace_H_Base(replace_G_Base(comm_313_expr, r'\eta'), "f")/A_tensor
dW_ds_323_expr = replace_H_Base(replace_G_Base(comm_323_expr, r'\eta'), r'\Gamma')/A_tensor/4
dW_ds_333_expr = replace_H_Base(replace_G_Base(comm_333_expr, r'\eta'), "W")/A_tensor/36

# jupyterDisplay(dW_ds_133_expr, "[η_1B, W] → 3B:")
# jupyterDisplay(dW_ds_233_expr, "[η_2B, W] → 3B:")
# jupyterDisplay(dW_ds_313_expr, "[η_3B, f] → 3B:")
# jupyterDisplay(dW_ds_323_expr, "[η_3B, Γ] → 3B:")
# jupyterDisplay(dW_ds_333_expr, "[η_3B, W] → 3B:")


In [ ]:
print("dW^{abc}_{def}/ds flow equation contains:")
display(Latex(r'$$d W^{abc}_{def} =  $$'))
# [1,3]-3
print('commutator [1,3]-3, lambda_1B:')
jupyterDisplay(dW_ds_133_expr)
# [2,3]-3
print('commutator [2,3]-3, lambda_1B:')
jupyterDisplay(dW_ds_233_expr)
# [3,1]-3
print('commutator [3,1]-3, lambda_1B:')
jupyterDisplay(dW_ds_313_expr)
# [3,2]-3
print('commutator [3,2]-3, lambda_1B:')
jupyterDisplay(dW_ds_323_expr)
# [3,3]-3
print('commutator [3,3]-3, lambda_1B:')
jupyterDisplay(dW_ds_333_expr)


dW^{abc}_{def}/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,3]-3, lambda_1B:


<IPython.core.display.Latex object>

commutator [2,3]-3, lambda_1B:


...

<IPython.core.display.Latex object>

commutator [3,2]-3, lambda_1B:


<IPython.core.display.Latex object>

commutator [3,3]-3, lambda_1B:


<IPython.core.display.Latex object>

## $dW /ds$

$$\frac{d W_{123456}}{ds} = P(12/3)P(45/6)\sum_{p}\left(\eta_{3p45}\Gamma_{126p} - \Gamma_{3p45}\eta_{126p}\right) $$
$$+ P(12/3)\sum_{p}\left(\eta_{3p}W_{12p456} - f_{3p}\eta_{12p456}\right) - P(45/6)\sum_{p}\left(\eta_{p6}W_{12345p} - f_{p6}\eta_{12345p}\right) $$
$$+ \frac{1}{2}P(12/3)\sum_{pq}\left(\bar{n}_{p}\bar{n}_{q} - n_{p}n_{q}\right)\left(\eta_{12pq}W_{pq3456} - \Gamma_{12pq}\eta_{pq3456}\right) $$
$$- \frac{1}{2}P(45/6)\sum_{pq}\left(\bar{n}_{p}\bar{n}_{q} - n_{p}n_{q}\right)\left(\eta_{pq45}W_{123pq6} - \Gamma_{pq45}\eta_{123pq6}\right) $$
$$+ P(12/3)P(45/6)\sum_{pq}\left(\bar{n}_{p}n_{q} - n_{p}\bar{n}_{q}\right)\left(\eta_{3pq6}W_{12q45p} - \Gamma_{3pq6}\eta_{12q45p}\right) $$
$$+ \frac{1}{6}\sum_{pqr}\left(n_{p}n_{q}n_{r} + \bar{n}_{p}\bar{n}_{q}\bar{n}_{r}\right)\left(\eta_{123pqr}W_{pqr456} - W_{123pqr}\eta_{pqr456}\right) $$
$$+ \frac{1}{2}P(12/3)P(45/6)\sum_{pqr}\left(\bar{n}_{p}\bar{n}_{q}n_{r} + n_{p}n_{q}\bar{n}_{r}\right)\left(\eta_{pq345r}W_{12rpq6} - W_{pq345r}\eta_{12rpq6}\right),$$

---
## 5. Summary: Complete SR-IMSRG(3) Flow Equations

### Key SR Differences from MR
- No $\lambda_{2B}$ or $\lambda_{3B}$ density matrices — only $\lambda_{1B}$ ($\lambda^a_b$) and $\xi^a_b$ appear
- Many commutators that would produce multi-body $\lambda$ in MR are zero in SR
- The contraction rule $|m-n| \leq k < m+n$ restricts which body ranks can be produced

**Reference**: H. Heinz, et.al **In-medium similarity renormalization group with three-body operators**